In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader,Dataset
from laplace import Laplace
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)


In [ ]:

n_epochs = 200
verbose_option = True

# Regression for Naval Plant Maintenance

Load dataset

In [ ]:
class DummyDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
npm = pd.read_csv('navalplantmaintenance.csv',header=None)
npm_train, npm_test = train_test_split(npm,test_size=0.25,random_state=42)
npm_train_np = npm_train.to_numpy()
npm_test_np = npm_test.to_numpy()
x_train_np = npm_train_np[:,:16]
x_test_np = npm_test_np[:,:16]
y_train_np = npm_train_np[:,17]
y_test_np = npm_test_np[:,17]
x_mu = x_train_np.mean(axis=0)
x_sigma = x_train_np.std(axis=0)
y_mu=y_train_np.mean(axis=0)
y_sigma=y_train_np.std(axis=0)
def scale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return (x-x_mu)/x_sigma
def unscale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return x_sigma*x+x_mu
x_train_np_z = scale(x_train_np,x_mu,x_sigma)
y_train_np_z = scale(y_train_np,y_mu,y_sigma)
x_test_np_z = scale(x_test_np,x_mu,x_sigma)
y_test_np_z = scale(y_test_np,y_mu,y_sigma)
x_train_t_z = torch.FloatTensor(x_train_np_z)
y_train_t_z = torch.FloatTensor(y_train_np_z)
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)

In [ ]:
train_t_z_dataset = DummyDataset(x_train_t_z,y_train_t_z)
train_t_z_dataloader = DataLoader(train_t_z_dataset)

1. Using PyTorch, perform variational inference using MC Dropout for a non-linear Gaussian
prediction model with heteroscedastic uncertainty for the regression dataset.

In [ ]:
def nlls(y, mu, std):
    return torch.square(y - mu)/(2.0*torch.square(std))+torch.log(std)
n_train_examples = x_train_np_z.shape[0]

class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = torch.nn.Linear(inputSize, hiddenSize)
        self.layer2 = torch.nn.Linear(hiddenSize, hiddenSize)
        self.linear_mu = torch.nn.Linear(hiddenSize, outputSize)
        self.linear_sigma = torch.nn.Linear(hiddenSize, outputSize)

    def forward(self, x):
        h1 = #TODO: Run the first layer using dropout with p_drop = 0.8 and ReLU
        h2 = #TODO: Run the second layer using dropout with p_drop = 0.8 and ReLU
        mu = self.linear_mu(h2)
        sigma = torch.nn.functional.softplus(self.linear_sigma(h2))
        return mu, sigma

    def L2reg(self):
        l2reg_sum = 0.0
        l2reg_sum += torch.square(self.layer1.weight).sum()
        l2reg_sum += torch.square(self.layer1.bias).sum()
        l2reg_sum += torch.square(self.layer2.weight).sum()
        l2reg_sum += torch.square(self.layer2.bias).sum()
        l2reg_sum += torch.square(self.linear_mu.weight).sum()
        l2reg_sum += torch.square(self.linear_mu.bias).sum()
        l2reg_sum += torch.square(self.linear_sigma.weight).sum()
        l2reg_sum += torch.square(self.linear_sigma.bias).sum()
        return l2reg_sum

In [ ]:
model = nn(x_train_np_z.shape[1],50, 1)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)
l2_coeff = 1e-4
for i in range(50):
    mu,s= model(x_train_t_z)
    nll_loss = nlls(y_train_t_z, mu, s).mean()
    loss = nll_loss + l2_coeff * model.L2reg()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if verbose_option: print(i, loss)

2. Compute the mean predictions for 20 MC sampled models

In [ ]:
mc_samples = 20
n_test_examples = y_test_np.shape[0]
y_test_mus_z = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    results = model(x_test_t_z)
    y_test_mus_z[i] = #TODO: Get the predicted means for one sampled parameter vector

3. Compute the Mean Squared Error (MSE) for the variational distribution for the non-linear heteroscedastic regression model for the test data using 20 MC samples and Bayesian model averaging.

In [ ]:
y_test_mu_z = #TODO: Compute the mean predictions using Bayesian model averaging
y_test_mu = unscale(y_test_mu_z,y_mu,y_sigma)
print('MSE:', mean_squared_error(y_test_np, y_test_mu))

4. Compute the epistemic uncertainties for each regression test example.

In [ ]:
y_test_mus = unscale(y_test_mus_z,y_mu,y_sigma)
y_test_epistemic = #TODO: Compute the epistemic uncertainties

In [ ]:
y_test_epistemic

5. Select the best test example to add to the training set using epistemic-uncertainity-based active learning

In [ ]:
selected_x_test = #TODO: Get the index of the best test example to select for active learning

In [ ]:
selected_x_test

# Classification for Ship Detection


Load Ship Detection Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torch.utils.data import random_split
from torchvision.transforms.functional import resize
from sklearn import preprocessing
import numpy as np
from pathlib import Path
import torchmetrics

ROOT_PATH = "shipsnet/shipsnet"
LR = 1e-4
IMG_SIZE = [80]

tensor_size = IMG_SIZE[0]**2 * 3

def max_scaling(image):
    image = image / 255.0
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    return image

def normalize_img(image):
    means = torch.Tensor([[[105.0385]],[[108.1886]],[[ 94.9558]]])
    stds = torch.Tensor([[[48.4294]],[[40.0104]],[[38.6445]]])
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    image = image - means
    image = image / stds
    return image

#https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
class ShipDataset(Dataset):
    def __init__(self, root_path, transform = None):
        self.root_path = Path(root_path)
        self.files = list(self.root_path.rglob("*/*"))
        self.classes = list(set([int(entry.parts[-1]) for entry in self.root_path.rglob("*") if Path(entry).is_dir()]))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = read_image(str(self.files[idx]))
        label = int(self.files[idx].parts[-2])
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset  = ShipDataset(ROOT_PATH, transform = normalize_img,)
n_classes = len(full_dataset.classes)
n_examples = len(full_dataset)
train_dataset, test_dataset = random_split(full_dataset, [int(0.8*float(n_examples)), int(0.2*float(n_examples))])
n_train_examples = len(train_dataset)
train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset))

criterion = torch.nn.BCELoss(reduce='mean')
accuracy = torchmetrics.classification.BinaryAccuracy()


In [ ]:
n_train_examples = len(train_dataset)

6. Using PyTorch, perform variational inference using Concrete Dropout for a non-linear Bernoulli prediction model for the binary classification dataset

In [ ]:
class ConcreteDropout(torch.nn.Module):

    """Concrete Dropout.

    Implementation of the Concrete Dropout module as described in the
    'Concrete Dropout' paper: https://arxiv.org/pdf/1705.07832
    """

    def __init__(self,
                 weight_regulariser: float,
                 dropout_regulariser: float,
                 init_min: float = 0.1,
                 init_max: float = 0.1) -> None:

        """Concrete Dropout.

        Parameters
        ----------
        weight_regulariser : float
            Weight regulariser term.
        dropout_regulariser : float
            Dropout regulariser term.
        init_min : float
            Initial min value.
        init_max : float
            Initial max value.
        """

        super().__init__()

        self.weight_regulariser = weight_regulariser
        self.dropout_regulariser = dropout_regulariser

        init_min = np.log(init_min) - np.log(1.0 - init_min)
        init_max = np.log(init_max) - np.log(1.0 - init_max)

        self.p_logit = torch.nn.parameter.Parameter(torch.empty(1).uniform_(init_min, init_max))
        self.p = torch.sigmoid(self.p_logit)

        self.regularisation = 0.0

    def forward(self, x: torch.Tensor, layer: torch.nn.Module) -> torch.Tensor:

        """Calculates the forward pass.

        The regularisation term for the layer is calculated and assigned to a
        class attribute - this can later be accessed to evaluate the loss.

        Parameters
        ----------
        x : Tensor
            Input to the Concrete Dropout.
        layer : nn.Module
            Layer for which to calculate the Concrete Dropout.

        Returns
        -------
        Tensor
            Output from the dropout layer.
        """

        output = layer(self._concrete_dropout(x))

        sum_of_squares = 0
        for param in layer.parameters():
            sum_of_squares += torch.sum(torch.pow(param, 2))

        weights_reg = self.weight_regulariser * sum_of_squares / (1.0 - self.p)

        dropout_reg = self.p * torch.log(self.p)
        dropout_reg += (1.0 - self.p) * torch.log(1.0 - self.p)
        dropout_reg *= self.dropout_regulariser * x[0].numel()

        self.regularisation = weights_reg + dropout_reg

        return output

    def _concrete_dropout(self, x: torch.Tensor) -> torch.Tensor:

        """Computes the Concrete Dropout.

        Parameters
        ----------
        x : Tensor
            Input tensor to the Concrete Dropout layer.

        Returns
        -------
        Tensor
            Outputs from Concrete Dropout.
        """

        eps = 1e-7
        tmp = 0.1

        self.p = torch.sigmoid(self.p_logit)
        u_noise = torch.rand_like(x)

        drop_prob = (torch.log(self.p + eps) -
                     torch.log(1 - self.p + eps) +
                     torch.log(u_noise + eps) -
                     torch.log(1 - u_noise + eps))

        drop_prob = torch.sigmoid(drop_prob / tmp)

        random_tensor = 1 - drop_prob
        retain_prob = 1 - self.p

        x = torch.mul(x, random_tensor) / retain_prob

        return x

In [ ]:
w = 1./(100.*float(n_train_examples))
d = 1./float(n_train_examples)
class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = torch.nn.Linear(inputSize, hiddenSize)
        self.layer2 = torch.nn.Linear(hiddenSize, hiddenSize)
        self.layer3 = torch.nn.Linear(hiddenSize, outputSize)
        self.cd1 = #TODO: Call the constructor for ConcreteDropout
        self.cd2 = #TODO: Call the constructor for ConcreteDropout
        self.cd3 = #TODO: Call the constructor for ConcreteDropout
        self.relu = torch.nn.ReLU()
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        h1 = self.cd1(x, torch.nn.Sequential(self.layer1,self.relu))
        h2 = self.cd2(h1, torch.nn.Sequential(self.layer2,self.relu))
        l = self.cd3(h2, self.layer3)
        return torch.nn.functional.sigmoid(l)

    def reg(self):
        reg = 0.0
        reg += self.cd1.regularisation
        reg += self.cd2.regularisation
        reg += self.cd3.regularisation
        return reg

In [ ]:
model = nn(tensor_size, 200, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(50):
    for data, label in train_dataloader:
        optimizer.zero_grad()
        p = model(data)
        nll_loss = criterion(p.squeeze(),label.float().squeeze())
        loss = #TODO: Get variational loss for the model
        acc = accuracy(p.squeeze(), label.float().squeeze())
        loss.backward()
        optimizer.step()
        if verbose_option: print(epoch, loss, acc)

7.  Compute the predicted probabilities and entropy predictions for 20 MC sampled models

In [ ]:
mc_samples = 20
n_test_examples = len(test_dataset)
y_test_probs = np.zeros([mc_samples,n_test_examples,1])
y_test_entropies = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    for data, label in test_dataloader:
        results = model(data).detach().numpy()
        y_test_probs[i] = #TODO: Get the predicted means for one sampled parameter vector
        y_test_entropies[i] += #TODO: Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector

8. Compute the Bayesian model averaging predictions for each classification test example.

In [ ]:
y_test_probs_avg = #TODO: Compute Bayesian model averaging predictions

In [ ]:
y_test_probs_avg

9. Compute the aleatoric uncertainty for each classification test example.

In [ ]:
y_test_aleatoric  = #TODO: Computer aleatoric uncertainty

In [ ]:
y_test_aleatoric

10. Compute the epistemic uncertainty for each classification test example.

In [ ]:
y_test_uncertainty = y_test_probs_avg * -1.*np.log(y_test_probs_avg) + (1. - y_test_probs_avg) * -1.*np.log(1. -y_test_probs_avg) #TODO: Compute the total uncertainity
y_test_epistemic = #TODO: Compute the epistemic uncertainty

In [ ]:
y_test_epistemic

In [ ]:
from sklearn.metrics import classification_report

for data, label in test_dataloader:
    print(classification_report(label.flatten(), y_test_probs_avg.round().flatten()))